In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.sparse import hstack
# Feature processing
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer

columns = [
    "label", "statement", "subject", "speaker", "speaker_job",
    "state_info", "party_affiliation", "barely_true_counts", "false_counts",
    "half_true_counts", "mostly_true_counts", "pants_on_fire_counts", "context"
]
train_df = pd.read_csv("train.tsv", sep='\t', header=None, names=columns)
valid_df = pd.read_csv("valid.tsv", sep='\t', header=None, names=columns)
test_df  = pd.read_csv("test.tsv",  sep='\t', header=None, names=columns)

# Encode labels
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df['label'])
y_valid = label_encoder.transform(valid_df['label'])
y_test  = label_encoder.transform(test_df['label'])

# TF-IDF
tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=5000)
X_train_text = tfidf.fit_transform(train_df['statement'])
X_valid_text = tfidf.transform(valid_df['statement'])
X_test_text  = tfidf.transform(test_df['statement'])

# Metadata (categorical)
meta_cols = ['party_affiliation', 'speaker_job']
encoder = OneHotEncoder(handle_unknown='ignore')
train_meta = encoder.fit_transform(train_df[meta_cols])
valid_meta = encoder.transform(valid_df[meta_cols])
test_meta  = encoder.transform(test_df[meta_cols])


# Credit history fix
credit_cols = ["pants_on_fire_counts", "false_counts", "barely_true_counts", "half_true_counts", "mostly_true_counts"]
for df in [train_df, valid_df, test_df]:
    df[credit_cols] = df[credit_cols].fillna(0).astype(int)

def correct_credit(row):
    counts = row[credit_cols].values.astype(int).copy()
    label = row['label']
    if label == "pants-fire": counts[0] = max(0, counts[0] - 1)
    elif label == "false": counts[1] = max(0, counts[1] - 1)
    elif label == "barely-true": counts[2] = max(0, counts[2] - 1)
    elif label == "half-true": counts[3] = max(0, counts[3] - 1)
    elif label == "mostly-true": counts[4] = max(0, counts[4] - 1)
    return counts

train_df['credit_vector'] = train_df.apply(correct_credit, axis=1)
valid_df['credit_vector'] = valid_df.apply(correct_credit, axis=1)
test_df['credit_vector']  = test_df.apply(correct_credit, axis=1)

credit_train = np.stack(train_df['credit_vector'].values).astype(float)
credit_valid = np.stack(valid_df['credit_vector'].values).astype(float)
credit_test  = np.stack(test_df['credit_vector'].values).astype(float)

scaler = MinMaxScaler()
credit_train = scaler.fit_transform(credit_train)
credit_valid = scaler.transform(credit_valid)
credit_test  = scaler.transform(credit_test)

# Final features
X_train_ext = hstack([X_train_text, train_meta, credit_train])
X_valid_ext = hstack([X_valid_text, valid_meta, credit_valid])
X_test_ext  = hstack([X_test_text,  test_meta,  credit_test])

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_ext, y_train)
val_preds = lr_model.predict(X_valid_ext)
print("Logistic Regression:\n", accuracy_score(y_valid, val_preds))
print(classification_report(y_valid, val_preds, target_names=label_encoder.classes_))


Logistic Regression:
 0.2749221183800623
              precision    recall  f1-score   support

 barely-true       0.23      0.17      0.20       237
       false       0.29      0.32      0.31       263
   half-true       0.25      0.29      0.27       248
 mostly-true       0.30      0.31      0.30       251
  pants-fire       0.44      0.29      0.35       116
        true       0.23      0.26      0.24       169

    accuracy                           0.27      1284
   macro avg       0.29      0.27      0.28      1284
weighted avg       0.28      0.27      0.27      1284



In [19]:
import joblib

# Save trained Logistic Regression model
joblib.dump(lr_model, "lr_model.pkl")

# Save TF-IDF vectorizer
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

# Save OneHotEncoder
joblib.dump(encoder, "onehot_encoder.pkl")

# Save MinMaxScaler for credit history
joblib.dump(scaler, "minmax_scaler.pkl")


['minmax_scaler.pkl']